# Figure 3 — 20× DA subtype/depth, compartment, and state

This notebook follows the agreed priority order. Every section declares the calculation, saves its table, and renders a reviewable panel. Populations are DAT superficial, TH superficial, and TH deep; compartments are somas and process-only grouped aggregates.

## 0. Configuration and scientific decision points

Defaults are explicit rather than buried in helper functions. Please review `F0_MODE`, temporal windows, reliability repeats, pupil coverage, and the reliability gate before final figures.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
HERE = Path.cwd().resolve()
REPO = next((p for p in (HERE, *HERE.parents) if (p / 'analysis').is_dir()), None)
if REPO is None: raise RuntimeError('Could not locate the ODyn-analysis repository root.')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
MANIFEST = REPO / 'analysis/stage0/ketxyl_16odor_session_manifest.csv'
IMAGING_ROOT = Path('/Volumes/MossLab/ImagingData')
OUTPUT = REPO / 'analysis/figures/figure3/outputs/cellular_20x'
OUTPUT.mkdir(parents=True, exist_ok=True)
POPULATIONS = ('somas', 'processes')
F0_MODE = 'log2_post_pre_detrended'  # within-unit ratio; F is not background-subtracted
ODOR_WINDOW = (0., 4.)
EARLY_WINDOW = (0., 2.)
LATE_WINDOW = (2., 4.)
RELIABILITY_REPEATS = 50
MIN_PUPIL_COVERAGE = .70
MIN_TRIALS_PER_ODOR = 4  # primary geometry eligibility, within session × state
SENSITIVITY_MIN_TRIALS = 5
EXAMPLE_GROUPS = {'DAT superficial': 202, 'TH superficial': 199, 'TH deep': 203}
TEMPORAL_HEATMAP_LIMITS = (-3, 6)  # same asymmetric z range used by 20x QC
print('Repository:', REPO)
print('Output:', OUTPUT)

## Input inventory

Only current final grouped 20× products enter. Missing files remain visible in the saved inventory.

In [ ]:
from analysis.figures.session_data import available_sessions
inventory = pd.DataFrame(available_sessions(MANIFEST, IMAGING_ROOT, objective='20x'))
inventory = inventory[inventory.population.str.startswith(('TH-', 'DAT-'))].copy()
inventory['line'] = inventory.population.str.split('-').str[0]
inventory['cohort'] = inventory.line + ' ' + inventory.depth_class
inventory.to_csv(OUTPUT / 'session_inventory.csv', index=False)
inputs = inventory.loc[inventory.available].to_dict('records')
display(inventory[['group_id', 'mouse', 'cohort', 'available', 'grouped_path']])

## Plotting helper: sessions nested within mice

ROI-level values are reduced to session medians; sessions are then averaged within mouse. Thick trajectories show the median mouse estimate.

In [ ]:
COLORS = {'DAT superficial': '#6a51a3', 'TH superficial': '#2b8cbe', 'TH deep': '#e34a33'}
def paired_state_panel(table, metric, title, ylabel, path):
    session = table.groupby(['group_id','mouse','cohort','compartment','state'], as_index=False)[metric].median()
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True, constrained_layout=True)
    for ax, compartment in zip(axes, POPULATIONS):
        selected_compartment = session[session.compartment == compartment]
        for cohort, color in COLORS.items():
            selected = selected_compartment[selected_compartment.cohort == cohort]
            for _, values in selected.groupby('group_id'):
                values = values.set_index('state')[metric]
                if {'pre','post'}.issubset(values.index): ax.plot([0,1],[values.pre,values.post],color=color,alpha=.2,lw=.8)
            mouse = selected.groupby(['mouse','state'])[metric].mean().unstack()
            if {'pre','post'}.issubset(mouse.columns): ax.plot([0,1],mouse[['pre','post']].median(),color=color,marker='o',lw=2.5,label=cohort)
        ax.set(xticks=[0,1],xticklabels=['awake','ket/xyl'],title=compartment,ylabel=ylabel)
    axes[0].legend(frameon=False,fontsize=7)
    fig.suptitle(title); fig.savefig(path,dpi=200); plt.show(); print('Saved:',path)

# Priority 1 — tonic F0 by population and compartment

**Decision:** use the within-unit log2 post/pre ratio of the detrended baseline fluorescence stored by QC. This avoids between-line expression comparisons but remains sensitive to background fluorescence because no background-subtracted F0 is stored.

In [ ]:
from analysis.figures.cellular_20x import load_population, tonic_table
tonic_parts=[]; tonic_failures=[]
for row in tqdm(inputs,desc='20x tonic F0',unit='session'):
    for population in POPULATIONS:
        try: tonic_parts.append(tonic_table(load_population(row['grouped_path'],population),row,population))
        except Exception as error: tonic_failures.append((row['group_id'],population,str(error)))
tonic=pd.concat(tonic_parts,ignore_index=True)
tonic.to_csv(OUTPUT/'tonic_f0_units.csv',index=False)
print('Failures:',tonic_failures)
display(tonic.head())

### Panel 3A — tonic state change

In [ ]:
tonic_effect=tonic.drop_duplicates(['group_id','compartment','unit_id'])
fig,axes=plt.subplots(1,2,figsize=(8,3.5),sharey=True,constrained_layout=True)
rng=np.random.default_rng(0)
for ax,compartment in zip(axes,POPULATIONS):
    session=tonic_effect[tonic_effect.compartment==compartment].groupby(['group_id','mouse','cohort'],as_index=False).f0_log2_post_pre.median()
    for x,cohort in enumerate(COLORS):
        selected=session[session.cohort==cohort]
        ax.scatter(x+rng.uniform(-.08,.08,len(selected)),selected.f0_log2_post_pre,color=COLORS[cohort],alpha=.65,s=24)
        mouse=selected.groupby('mouse').f0_log2_post_pre.mean(); ax.plot(x,np.median(mouse),'_',color=COLORS[cohort],ms=22,mew=3)
    ax.axhline(0,color='.7',lw=.7); ax.set(xticks=range(3),xticklabels=COLORS,rotation=20,title=compartment,ylabel='log2 ket/xyl ÷ awake F0')
panel=OUTPUT/'panel_3A_tonic_f0.png'; fig.savefig(panel,dpi=200); plt.show(); print('Saved:',panel)

# Priority 2 — raw positive and negative temporal AUC

AUC is calculated from the median waveform for each unit × odor × state. Positive and negative AUC are raw zero-rectified z-score integrals and are never forced onto a common threshold-excess scale.

In [ ]:
from analysis.figures.cellular_20x import TemporalWindows, temporal_feature_table
WINDOWS=TemporalWindows(odor=ODOR_WINDOW,early=EARLY_WINDOW,late=LATE_WINDOW)
temporal_parts=[]; temporal_failures=[]
for row in tqdm(inputs,desc='20x temporal features',unit='session'):
    for population in POPULATIONS:
        try: temporal_parts.append(temporal_feature_table(load_population(row['grouped_path'],population),row,population,windows=WINDOWS,reducer='median'))
        except Exception as error: temporal_failures.append((row['group_id'],population,str(error)))
temporal=pd.concat(temporal_parts,ignore_index=True)
temporal.to_csv(OUTPUT/'temporal_unit_odor_features.csv',index=False)
print('Failures:',temporal_failures); display(temporal.head())

### Panels 3B–C — excitation and suppression AUC state effects

In [ ]:
paired_state_panel(temporal,'positive_auc_z_s','Positive odor-response AUC','positive AUC (z·s)',OUTPUT/'panel_3B_positive_auc.png')
paired_state_panel(temporal,'negative_auc_z_s','Negative odor-response AUC magnitude','negative AUC magnitude (z·s)',OUTPUT/'panel_3C_negative_auc.png')

# Priority 3 — signed lifetime sparseness

Lifetime sparseness is computed separately across the analog positive-AUC and negative-AUC odor profiles of each unit. This version is threshold-free; responder breadth remains a separate mineral-oil-tail analysis.

In [ ]:
from analysis.figures.cellular_20x import specificity_table
specificity=specificity_table(temporal)
specificity.to_csv(OUTPUT/'signed_auc_specificity.csv',index=False)
display(specificity.head())

### Panels 3D–E — excitatory and suppressive lifetime sparseness

In [ ]:
paired_state_panel(specificity,'excitation_auc_lifetime_sparseness','Excitatory lifetime sparseness','sparseness',OUTPUT/'panel_3D_excitation_sparseness.png')
paired_state_panel(specificity,'suppression_auc_lifetime_sparseness','Suppressive lifetime sparseness','sparseness',OUTPUT/'panel_3E_suppression_sparseness.png')

### Responder breadth from asymmetric mineral-oil tails

Breadth is kept separate from analog AUC sparseness. Excited and suppressed calls use their own empirical blank tails; the resulting threshold-excess magnitudes are not compared to one another.

In [ ]:
from analysis.figures.session_data import load_grouped
from analysis.figures.summaries import signed_session_tables
breadth_parts=[]; breadth_failures=[]
for row in tqdm(inputs,desc='20x responder breadth',unit='session'):
    for population in POPULATIONS:
        try:
            session=load_grouped(row,row['grouped_path'],population=population)
            units,_=signed_session_tables(session,blank_odor=0,tail_probability=.01,reducer='median')
            table=pd.DataFrame(units); table['cohort']=row['cohort']; table['compartment']=population; breadth_parts.append(table)
        except Exception as error: breadth_failures.append((row['group_id'],population,str(error)))
breadth=pd.concat(breadth_parts,ignore_index=True)
breadth.to_csv(OUTPUT/'asymmetric_blank_responder_breadth.csv',index=False)
print('Failures:',breadth_failures); display(breadth.head())

### Panel 3F — excitatory and suppressive breadth

In [ ]:
paired_state_panel(breadth,'excitation_breadth','Excitation breadth','fraction of odors',OUTPUT/'panel_3F_excitation_breadth.png')
paired_state_panel(breadth,'suppression_breadth','Suppression breadth','fraction of odors',OUTPUT/'panel_3F_suppression_breadth.png')

# Priority 4 — within-odor and tuning reliability

Unit reliability is repeated split-half correlation of odor-tuning profiles. Population-pattern reliability is repeated split-half cosine similarity for the same odor. Reliability is evaluated before interpreting geometry.

In [ ]:
from analysis.figures.cellular_20x import reliability_tables
reliability_unit_parts=[]; reliability_odor_parts=[]; reliability_failures=[]
for row in tqdm(inputs,desc='20x reliability',unit='session'):
    for population in POPULATIONS:
        try:
            unit,odor=reliability_tables(load_population(row['grouped_path'],population),row,population,repeats=RELIABILITY_REPEATS,seed=int(row['group_id']),windows=WINDOWS)
            reliability_unit_parts.append(unit); reliability_odor_parts.append(odor)
        except Exception as error: reliability_failures.append((row['group_id'],population,str(error)))
reliability_units=pd.concat(reliability_unit_parts,ignore_index=True)
reliability_odors=pd.concat(reliability_odor_parts,ignore_index=True)
reliability_units.to_csv(OUTPUT/'unit_tuning_reliability.csv',index=False)
reliability_odors.to_csv(OUTPUT/'odor_population_reliability.csv',index=False)
print('Failures:',reliability_failures); display(reliability_units.head())

### Panels 3F–G — tuning and population-pattern reliability

In [ ]:
paired_state_panel(reliability_units,'tuning_reliability_signed','Unit odor-tuning reliability','split-half r',OUTPUT/'panel_3F_tuning_reliability.png')
paired_state_panel(reliability_odors,'population_pattern_reliability','Within-odor population-pattern reliability','split-half cosine similarity',OUTPUT/'panel_3G_population_reliability.png')

# Priority 5 — qualitative temporal organization

Temporal smoothing is shown with representative population traces rather than promoted to a scalar metric. The example session and odor are declared in the configuration cell. Shading is the interquartile range across units after taking each unit's median across trials. This panel is qualitative and does not support inference.

### Panel 3H — representative awake and anesthetized traces

In [ ]:
from matplotlib.colors import TwoSlopeNorm
from analysis.seg_10x.grouped_qc import temporal_unit_odor_heatmaps
# Each heatmap stacks every ROI once within every odor. Trials are reduced with
# the median. ROI order is determined separately for each odor from the awake
# response latency, then held fixed in the anesthetized panel. Thus changes in
# sign, latency, duration, and temporal smoothness remain directly comparable.
fig,axes=plt.subplots(3,4,figsize=(15,10),sharex=True,constrained_layout=True)
norm=TwoSlopeNorm(vmin=TEMPORAL_HEATMAP_LIMITS[0],vcenter=0,vmax=TEMPORAL_HEATMAP_LIMITS[1])
image=None
for row_index,(cohort,group_id) in enumerate(EXAMPLE_GROUPS.items()):
    row=next(item for item in inputs if int(item['group_id'])==group_id)
    for population_index,population in enumerate(POPULATIONS):
        data=load_population(row['grouped_path'],population)
        frame_rate=1/np.nanmedian(np.diff(data['time_s']))
        odor_on=np.full(data['z'].shape[1],np.argmin(np.abs(data['time_s'])),int)
        odor_off=odor_on+int(round(4*frame_rate))
        atlas=temporal_unit_odor_heatmaps(
            data['z'],odor_ids=data['odor_id'],states=data['state'],
            state_levels=data['state_levels'],odor_on_frames=odor_on,
            odor_off_frames=odor_off,frame_rate=frame_rate,reducer='median',
            sort_by='latency')
        for state_index,state in enumerate(('pre','post')):
            column=population_index*2+state_index; ax=axes[row_index,column]
            values=atlas['heatmaps'][state]
            image=ax.imshow(values,aspect='auto',cmap='RdBu_r',norm=norm,
                extent=(atlas['time_s'][0],atlas['time_s'][-1],len(values)-.5,-.5),
                interpolation='nearest')
            ax.axvline(0,color='black',lw=.7); ax.axvline(atlas['odor_offset_s'],color='black',lw=.7,ls='--')
            for boundary in atlas['odor_boundaries']: ax.axhline(boundary,color='black',lw=.25,alpha=.65)
            ax.set(title=f'{cohort} — {population} — {state}',yticks=atlas['odor_centers'],
                yticklabels=[str(int(value)) for value in atlas['odors']],ylabel='odor ID')
for ax in axes[-1]: ax.set_xlabel('time from odor onset (s)')
fig.colorbar(image,ax=axes,label='median trial response (z)',shrink=.72)
fig.suptitle('Qualitative temporal atlases: all ROIs × all odors')
panel=OUTPUT/'panel_3H_qualitative_temporal_heatmaps.png'; fig.savefig(panel,dpi=200); plt.show(); print('Saved:',panel)

# Priority 6 — matched soma versus grouped processes

Only shared curated `g…` IDs are paired. Unmatched somas and process-only singletons do not enter this comparison.

In [ ]:
from analysis.figures.cellular_20x import matched_compartment_table
matched=matched_compartment_table({'somas':temporal[temporal.compartment=='somas'],'processes':temporal[temporal.compartment=='processes']},value_columns=['positive_auc_z_s','negative_auc_z_s'])
matched.to_csv(OUTPUT/'matched_soma_process_unit_odor.csv',index=False)
print('Matched unit × odor × state rows:',len(matched)); display(matched.head())

### Panel 3J — awake compartment identity and anesthesia × compartment effects

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(8,7),constrained_layout=True)
for row_index,(metric,title) in enumerate(zip(['positive_auc_z_s','negative_auc_z_s'],['Positive AUC','Negative AUC'])):
    session=matched.groupby(['group_id','mouse','cohort','state'],as_index=False)[[metric+'_soma',metric+'_process']].median()
    awake=session[session.state=='pre']
    for cohort,color in COLORS.items():
        selected=awake[awake.cohort==cohort]
        axes[row_index,0].scatter(selected[metric+'_soma'],selected[metric+'_process'],color=color,alpha=.65,label=cohort)
    pivot=session.pivot(index=['group_id','mouse','cohort'],columns='state',values=[metric+'_soma',metric+'_process']).dropna()
    delta_soma=pivot[(metric+'_soma','post')]-pivot[(metric+'_soma','pre')]
    delta_process=pivot[(metric+'_process','post')]-pivot[(metric+'_process','pre')]
    for cohort,color in COLORS.items():
        selected=pivot.index.get_level_values('cohort')==cohort
        axes[row_index,1].scatter(delta_soma[selected],delta_process[selected],color=color,alpha=.65)
    for ax in axes[row_index]:
        lo=min(ax.get_xlim()[0],ax.get_ylim()[0]); hi=max(ax.get_xlim()[1],ax.get_ylim()[1]); ax.plot([lo,hi],[lo,hi],color='.6',ls='--')
    axes[row_index,0].set(xlabel='soma',ylabel='matched processes',title=title+' — awake identity')
    axes[row_index,1].set(xlabel='Δ soma',ylabel='Δ matched processes',title=title+' — anesthesia × compartment')
axes[0,0].legend(frameon=False,fontsize=7)
panel=OUTPUT/'panel_3J_matched_soma_process.png'; fig.savefig(panel,dpi=200); plt.show(); print('Saved:',panel)

# Priority 7 — within-odor pupil/running association

Awake trial values are centered within odor before correlation. This asks whether trial-to-trial neural deviations covary with arousal deviations beyond the mean odor effect. It is associative and retains a motion-artifact caveat.

In [ ]:
from analysis.figures.arousal_20x import find_auxiliary, arousal_association_table
arousal_parts=[]; arousal_failures=[]
for row in tqdm(inputs,desc='20x arousal associations',unit='session'):
    aux=find_auxiliary(row,IMAGING_ROOT)
    if aux is None: arousal_failures.append((row['group_id'],'no auxiliary file')); continue
    for population in POPULATIONS:
        try: arousal_parts.append(arousal_association_table(row['grouped_path'],aux,row,population,windows=WINDOWS,minimum_pupil_coverage=MIN_PUPIL_COVERAGE))
        except Exception as error: arousal_failures.append((row['group_id'],population,str(error)))
arousal=pd.concat(arousal_parts,ignore_index=True) if arousal_parts else pd.DataFrame()
arousal.to_csv(OUTPUT/'within_odor_arousal_associations.csv',index=False)
print('Failures:',arousal_failures); display(arousal.head())

### Panel 3K — suppression versus pupil and running

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(8,3.5),sharey=True,constrained_layout=True)
for ax,metric,title in zip(axes,['suppression_vs_pupil_rho','suppression_vs_running_rho'],['Pupil dilation','Running speed']):
    session=arousal.groupby(['group_id','mouse','cohort','compartment'],as_index=False)[metric].median()
    for x,cohort in enumerate(COLORS):
        for offset,compartment in [(-.10,'somas'),(.10,'processes')]:
            selected=session[(session.cohort==cohort)&(session.compartment==compartment)]
            ax.scatter(np.full(len(selected),x+offset),selected[metric],color=COLORS[cohort],marker='o' if compartment=='somas' else '^',alpha=.65)
    ax.axhline(0,color='.7',lw=.7); ax.set(xticks=range(3),xticklabels=COLORS,rotation=20,title=title,ylabel='within-odor Spearman ρ')
panel=OUTPUT/'panel_3K_arousal_suppression.png'; fig.savefig(panel,dpi=200); plt.show(); print('Saved:',panel)

# Priority 8 — reliability gate before geometry

Reliability is retained as a continuous diagnostic. Primary geometry eligibility instead requires at least four trials for that odor within session × state; five trials is shown as a sensitivity threshold. Pairwise geometry requires both odors to pass.

In [ ]:
geometry_gate=reliability_odors.copy()
geometry_gate['eligible_primary']=geometry_gate.n_trials>=MIN_TRIALS_PER_ODOR
geometry_gate['eligible_sensitivity']=geometry_gate.n_trials>=SENSITIVITY_MIN_TRIALS
geometry_gate.to_csv(OUTPUT/'geometry_trial_reliability_gate.csv',index=False)
coverage=geometry_gate.groupby(['cohort','compartment','state'],as_index=False)[['eligible_primary','eligible_sensitivity']].mean()
fig,axes=plt.subplots(1,2,figsize=(8,3.5),sharey=True,constrained_layout=True)
for ax,compartment in zip(axes,POPULATIONS):
    selected=coverage[coverage.compartment==compartment]
    for x,cohort in enumerate(COLORS):
        values=selected[selected.cohort==cohort].set_index('state')
        if {'pre','post'}.issubset(values.index):
            ax.plot([x-.12,x+.12],values.loc[['pre','post'],'eligible_primary'],color=COLORS[cohort],marker='o',lw=2)
            ax.plot([x-.12,x+.12],values.loc[['pre','post'],'eligible_sensitivity'],color=COLORS[cohort],marker='.',ls='--',alpha=.6)
    ax.set(xticks=range(3),xticklabels=COLORS,rotation=20,title=compartment,ylabel='fraction odor × sessions eligible',ylim=(0,1.05))
panel=OUTPUT/'panel_3L_geometry_trial_gate.png'; fig.savefig(panel,dpi=200); plt.show(); print('Saved:',panel)
display(coverage)

## Locked analysis decisions

1. Main F0: detrended within-unit post/pre ratio.  
2. Temporal windows: early 0–2 s; late 2–4 s.  
3. Temporal smoothing: qualitative QC-style heatmaps only; no roughness claim.  
4. Soma/process: awake identity and anesthesia × compartment in adjacent panels.  
5. Pupil coverage: ≥70%.  
6. Neural-only sessions remain included; missing-aux sessions are omitted only from arousal.  
7. Geometry: ≥4 trials/odor/block primary; ≥5 sensitivity; reliability reported continuously.